<a href="https://colab.research.google.com/github/tsal4/data2000_labs/blob/main/homework/060_neural-networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework Assignment: Feed-Forward Neural Networks for Tabular Data

**Objective:** In this assignment, you will build, train, and evaluate a Feed-Forward Neural Network (Multi-Layer Perceptron) to process tabular data. You will use the **MovieLens 100k** dataset to predict user ratings for movies based on various features.

### Instructions
1. Run the provided starter code to load the dataset.
2. Complete the tasks outlined in the markdown cells below.
3. Ensure your code is well-commented and your plots are clearly labeled.
4. Answer any conceptual questions in a separate markdown block.

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

tf.keras.utils.set_random_seed(42)

In [2]:
dataset, ds_info = tfds.load(
    'movielens/100k-ratings',
    split='train',
    with_info=True,
)

# Display some dataset metadata
print(f"\nNumber of examples: {ds_info.splits['train'].num_examples}")
print(f"Dataset features: {list(ds_info.features.keys())}")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/movielens/100k-ratings/incomplete.HBRPOI_0.1.1/movielens-train.tfrecord*..…

Dataset movielens downloaded and prepared to /root/tensorflow_datasets/movielens/100k-ratings/0.1.1. Subsequent calls will reuse this data.

Number of examples: 100000
Dataset features: ['movie_id', 'movie_title', 'movie_genres', 'user_id', 'user_rating', 'timestamp', 'user_gender', 'bucketized_user_age', 'user_occupation_label', 'user_occupation_text', 'user_zip_code', 'raw_user_age']


In [29]:
unique_ratings = set(example['user_rating'].numpy() for example in dataset)
print(f"Unique ratings: {sorted(unique_ratings)}")

Unique ratings: [np.float32(1.0), np.float32(2.0), np.float32(3.0), np.float32(4.0), np.float32(5.0)]


I used Claude for this btw. I didn't think it would hurt to hurry up and use AI to figure out the rating scale because I didn't know how access the data.

## Task 1: Data Exploration
Before building a model, it is crucial to understand your tabular data.

**Your Task:**
1. Extract a batch of records from the dataset (e.g., using `dataset.take(5)`).
2. Print the features and their corresponding values to understand the structure of the data.
3. Identify the target variable (`user_rating`) and continuous/categorical features you might want to use.

In [3]:
records = []
for example in dataset.take(5):
    # Convert each tensor value to a numpy value
    record = {feature_name: feature_value.numpy() for feature_name, feature_value in example.items()}
    records.append(record)

df = pd.DataFrame(records)
df

,bucketized_user_age,movie_genres,movie_id,movie_title,raw_user_age,timestamp,user_gender,user_id,user_occupation_label,user_occupation_text,user_rating,user_zip_code
0,45.0,[7],b'357',"b""One Flew Over the Cuckoo's Nest (1975)""",46.0,879024327,True,b'138',4,b'doctor',4.0,b'53211'
1,25.0,"[4, 14]",b'709',b'Strictly Ballroom (1992)',32.0,875654590,True,b'92',5,b'entertainment',2.0,b'80525'
2,18.0,[4],b'412',"b'Very Brady Sequel, A (1996)'",24.0,882075110,True,b'301',17,b'student',4.0,b'55439'
3,50.0,"[5, 7]",b'56',b'Pulp Fiction (1994)',50.0,883326919,True,b'60',4,b'healthcare',4.0,b'06472'
4,50.0,"[10, 16]",b'895',b'Scream 2 (1997)',55.0,891409199,True,b'197',18,b'technician',3.0,b'75094'


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   bucketized_user_age    5 non-null      float32
 1   movie_genres           5 non-null      object 
 2   movie_id               5 non-null      object 
 3   movie_title            5 non-null      object 
 4   raw_user_age           5 non-null      float32
 5   timestamp              5 non-null      int64  
 6   user_gender            5 non-null      bool   
 7   user_id                5 non-null      object 
 8   user_occupation_label  5 non-null      int64  
 9   user_occupation_text   5 non-null      object 
 10  user_rating            5 non-null      float32
 11  user_zip_code          5 non-null      object 
dtypes: bool(1), float32(3), int64(2), object(6)
memory usage: 517.0+ bytes


## Task 2: Data Preprocessing


### Step 1: Extract Specific Features
First, we isolate the specific features we want our neural network to learn from. We extract `user_zip_code`, `user_gender`, `raw_user_age`, and `movie_genres` as our inputs, and `user_rating` as our target variable. We cast numerical and boolean fields to `tf.float32` for compatibility with TensorFlow.

In [5]:
def preprocess_data(features):
    inputs = {
        'user_zip_code': features['user_zip_code'],
        'user_gender': tf.cast(features['user_gender'], tf.float32),
        'raw_user_age': tf.cast(features['raw_user_age'], tf.float32),
        'movie_genres': features['movie_genres']
    }
    target = features['user_rating']
    return inputs, target

processed_dataset = dataset.map(preprocess_data)

### Step 2: Build Vocabularies and Normalize Data
Neural networks require numerical inputs, so we cannot pass raw strings or unscaled numbers directly.
- **Categorical Data**: We use a `Hashing` layer to convert high-cardinality string categories (like `user_zip_code`) into a fixed number of integer bins (e.g., 1000 bins). This avoids the need to compute and store a large vocabulary.
- **Multi-Categorical Data**: Since movies can have multiple genres, we use `IntegerLookup` with `multi_hot` encoding to represent genres as a multi-hot array. We can significantly speed up the .adapt() call by batching the dataset before passing it to the method. Processing the data in large batches (e.g., batch(10000)) allows TensorFlow to vectorize the operations and is much more efficient than processing one record at a time. Because the number of genres each movie has can vary (we call these ragged tensors), we will use the `.ragged_batch()` method.
- **Continuous Data**: We use a `Normalization` layer to scale continuous variables like `raw_user_age` so they have a mean of 0 and standard deviation of 1, which helps the network learn faster.

In [6]:
zip_code_hashing = tf.keras.layers.Hashing(num_bins=1000)

In [7]:
# Movie Genres Lookup (Multi-Hot)
genre_lookup = tf.keras.layers.IntegerLookup(output_mode='multi_hot')
genre_lookup.adapt(dataset.map(lambda x: x['movie_genres']).ragged_batch(10000))

In [8]:
# Age Normalization
age_normalization = tf.keras.layers.Normalization(axis=None)
age_normalization.adapt(
    dataset.map(lambda x: tf.cast(x['raw_user_age'], tf.float32)).batch(10000)
)

In [9]:
# Apply the transformations
def encode_features(inputs, target):
    encoded_inputs = {
        'user_zip_code': zip_code_hashing(inputs['user_zip_code']),
        'user_gender': inputs['user_gender'],
        'raw_user_age': age_normalization(inputs['raw_user_age']),
        'movie_genres': genre_lookup(inputs['movie_genres'])
    }
    return encoded_inputs, target

encoded_dataset = processed_dataset.map(encode_features)

### Step 3: Train/Test Split
To properly evaluate our model, we must test it on data it hasn't seen during training. We shuffle the dataset randomly and split it: 80% for training and 20% for testing.

In [10]:
num_examples = ds_info.splits['train'].num_examples
train_size = int(0.8 * num_examples)

encoded_dataset = encoded_dataset.shuffle(10000, seed=42)

train_dataset = encoded_dataset.take(train_size)
test_dataset = encoded_dataset.skip(train_size)

### Step 4: Batching and Prefetching
Finally, we group our data into batches (e.g., 32 records at a time) for more efficient memory usage during training.

We also use `.cache()` to keep the data in memory after the first epoch, and `.prefetch(tf.data.AUTOTUNE)` to allow the CPU to prepare the next batch of data while the GPU/CPU is simultaneously training the model on the current batch.

In [11]:
batch_size = 32

train_dataset = train_dataset.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)

print(f"Zip Code Bins: {zip_code_hashing.num_bins}")
print(f"Movie Genres Vocabulary Size: {genre_lookup.vocabulary_size()}")
print("Datasets successfully prepared, batched, and prefetched.")

Zip Code Bins: 1000
Movie Genres Vocabulary Size: 20
Datasets successfully prepared, batched, and prefetched.


## Task 3: Build the Feed-Forward Neural Network
Now you will design the architecture of your neural network for regression.

**Your Task:**
1. Use the Keras Functional API to build a feed-forward neural network.
2. Ensure your input layers correctly handle the shapes of the features you extracted. For categorical features that have been converted to integers (like `user_zip_code` which we hashed into bins), you must use an `Embedding` layer to map these integer indices into dense, continuous vectors. Remember to `Flatten` or pool the embedding output to remove the sequence dimension before concatenating it with your continuous and multi-hot features.
3. Add at least two `Dense` hidden layers with ReLU activation functions. Experiment with the number of neurons.
4. Add a final `Dense` output layer. Since this is a regression task predicting a single continuous value (`user_rating`), how many units should the output layer have, and what should the activation function be?

### Example Usage of Embedding and Flatten Layers
Here is a quick example of how you might process a categorical integer input using the Functional API before concatenating it with other features:

```python
# Define the input (shape is (1,) for a single
# categorical value per record)
categorical_input = tf.keras.Input(
    shape=(1,),
    name='example_cat_feature',
    dtype=tf.int64)

# Apply Embedding, in this case using the
# 1000 bins we created with our ZIP Code hash
# and outputting an embedding dimension of 16
embedding = tf.keras.layers.Embedding(
    input_dim=1000,
    output_dim=16)(categorical_input)
# The output shape here is (batch_size, 1, 16)

# Flatten to remove the extra sequence dimension
flattened_embedding = tf.keras.layers.Flatten()(embedding)
# The output shape is now (batch_size, 16), and is
# ready to be concatenated with the other inputs
```

### Model Code

In [39]:
#Using the example to embed and flatten the user_zip_code variable since it was categorical
user_zip_code = tf.keras.Input(
    shape=(1,),
    name='user_zip_code',
    dtype=tf.int64)

user_zip_code_embedding = tf.keras.layers.Embedding(
    input_dim=1000,
    output_dim=16)(user_zip_code)

user_zip_code_embedding_flattened = tf.keras.layers.Flatten()(user_zip_code_embedding)

In [40]:
#Doing the same thing for movie_genres since it was also categorical
movie_genres = tf.keras.Input(
    shape=(1,),
    name='movie_genres',
    dtype=tf.int64)

movie_genres_embedding = tf.keras.layers.Embedding(
    input_dim=1000,
    output_dim=16)(movie_genres)

movie_genres_embedding_flattened = tf.keras.layers.Flatten()(movie_genres_embedding)

In [41]:
#These are the rest of the inputs. They are float values so I use tf.float32
user_gender = tf.keras.layers.Input(
  shape=(1,),
  dtype=tf.float32,
  name='user_gender')

raw_user_age = tf.keras.layers.Input(
  shape=(1,),
  dtype=tf.float32,
  name='raw_user_age')

In [42]:
#Creating the input dictionary
inputs = {
    'user_zip_code_embedding_flattened': user_zip_code_embedding_flattened,
    'movie_genres_embedding_flattened': movie_genres_embedding_flattened,
    'user_gender': user_gender,
    'raw_user_age': raw_user_age
}

In [43]:
#I concatenate all the input layers into a single tensor
preprocessing_layers = tf.keras.layers.Concatenate()(
    [user_zip_code_embedding_flattened, movie_genres_embedding_flattened, user_gender, raw_user_age])

In [44]:
#Creating the first hidden layer with 50 neurons. Just randomly picked 40 to start with, nothing too high, but still a decent amount
hidden_layer_1 = tf.keras.layers.Dense(
    units=40,
    activation="relu",
    name="hidden_layer_1",
)(preprocessing_layers)

In [45]:
#Creating the second hidden layer with 20 neurons. Halved the neurons to start to narrow down predictions.
hidden_layer_2 = tf.keras.layers.Dense(
    units=40,
    activation="relu",
    name="hidden_layer_2",
)(hidden_layer_1)

In [46]:
#Creating the output layer with 5 neurons for each option (1-5 rating scale for the movies)
output = tf.keras.layers.Dense(
    units=5,
    activation="softmax",
    name="output",
)(hidden_layer_2)

#Also define the dictionary
outputs = {
    'output': output
}

**The output layer should have 5 units because the rating is 1-5. The activation function should be softmax, so that the sum of the probabilities are equal to one. This ensures that the model is giving clear, understandable probabilities for each possible outcome.**

## Task 4: Model Compilation, Training, and Evaluation

**Your Task:**
1. **Compile the model:** Use the Adam optimizer and Mean Squared Error (`mse`) or Mean Absolute Error (`mae`) for the loss function.
2. **Train the model:** Call `model.fit()` on your training dataset for at least 20 epochs. Use the test dataset as validation data.
3. **Plot training curves:** Extract the loss metric from the training history and plot it over the epochs.
4. **Evaluate:** Print out the final Mean Absolute Error (MAE) or Mean Squared Error (MSE) on the test dataset.

In [47]:
learning_rate = 0.01
epochs = 20
batch_size = 1000
label_name = "user_rating"

validation_split = 0.2

In [48]:
lin_model = tf.keras.Model(inputs=inputs, outputs=outputs)
lin_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    loss="mean_squared_error",
    metrics=[tf.keras.metrics.MeanSquaredError()]
)
lin_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:107: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'user_zip_code_embedding_flattened' mapping to value <KerasTensor shape=(None, 16), dtype=float32, sparse=False, ragged=False, name=keras_tensor_19> which has name 'keras_tensor_19'. Change the tensor name to 'user_zip_code_embedding_flattened' (via `Input(..., name='user_zip_code_embedding_flattened')`)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:107: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'movie_genres_embedding_flattened' mapping to value <KerasTensor shape=(None, 16), dtype=float32, sparse=False, ragged=False, name=keras_tensor_21> which has name 'keras_tensor_21'. Change the tensor name to 'movie_genres_embedding_flattened' (via `Input(...,

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ keras_tensor_19CLO… │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ keras_tensor_21CLO… │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_gender         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ raw_user_age        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 34)        │          0 │ keras_tensor_19C… │
│ (Concatenate)       │                   │            │ keras_tensor_21C… │
│                     │                   │            │ user_gender[1][0… │
│                     │                   │            │ raw_user_age[1][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hidden_layer_1      │ (None, 40)        │      1,400 │ concatenate_4[1]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hidden_layer_2      │ (None, 40)        │      1,640 │ hidden_layer_1[1… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 5)         │        205 │ hidden_layer_2[1… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,245 (12.68 KB)

 Trainable params: 3,245 (12.68 KB)

 Non-trainable params: 0 (0.00 B)